# Week 7 — Pure PyTorch LSTM Temporal Sequence Model (No Graph Features)

**Topic 04 (NT219 / NT2205):** *Agentic ML/DL for Metadata-Only TLS/SSH Threat Detection, Hunting & Response*

### Design & Compliance Checklist
- **Pure Temporal Sequence Modeling (Section 10 & 20)**: Uses `torch.nn.LSTM` on chronological flow sequences grouped by `src_host_token` and ordered by `flow_order_id`, with **zero graph features**.
- **Explicit Sequence Window Metadata**: Declares `seq_len` and `stride` explicitly in Dataset, Model, Metrics, and Checkpoint metadata.
- **Kaggle GPU Ready**: Automatically utilizes CUDA GPU on Kaggle and falls back cleanly to CPU locally, saving `state_dict` (`.pt`) + `_metadata.json`.

## Cell 1 — Configuration, Path Resolution & Reproducibility

In [1]:
import json
import math
import os
import platform
import random
import re
import sys
import time
import warnings
from collections.abc import Mapping
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, FrozenSet, List, Optional, Sequence, Tuple

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch

warnings.filterwarnings("ignore", category=UserWarning)

L3_L4_FEATURE_COLUMNS: List[str] = [
    "src_port_class", "dst_port", "proto", "in_bytes", "in_pkts", "out_bytes", "out_pkts",
    "tcp_flags", "client_tcp_flags", "server_tcp_flags", "flow_duration_ms", "duration_in",
    "duration_out", "min_ttl", "max_ttl", "longest_flow_pkt", "shortest_flow_pkt",
    "min_ip_pkt_len", "max_ip_pkt_len", "src_to_dst_second_bytes", "dst_to_src_second_bytes",
    "retransmitted_in_bytes", "retransmitted_in_pkts", "retransmitted_out_bytes",
    "retransmitted_out_pkts", "src_to_dst_avg_throughput", "dst_to_src_avg_throughput",
    "num_pkts_up_to_128_bytes", "num_pkts_128_to_256_bytes", "num_pkts_256_to_512_bytes",
    "num_pkts_512_to_1024_bytes", "num_pkts_1024_to_1514_bytes", "tcp_win_max_in",
    "tcp_win_max_out", "icmp_type", "icmp_ipv4_type", "is_tls_port", "is_ssh_port",
    "bytes_ratio", "pkts_ratio",
]

KAGGLE_DATASET_DIR = Path("/kaggle/input/datasets/vxdwkhoa/Dataset_NF/NF-UQ-NIDS-v2")
KAGGLE_CHECKPOINT_DIR = Path("/kaggle/working/Checkpoint")
LOCAL_DATASET_DIR = Path(r"D:\Crypto\DoAn\NF-UQ-NIDS-v2")
LOCAL_CHECKPOINT_DIR = Path(r"D:\Crypto\DoAn\Notebook\Checkpoint")
REPO_SAMPLE_PATH = Path(r"D:\Crypto\DoAn\topic04-agentic-tls-ssh-threat-detection\data\public_loaders\sample_data\nfuq_sample_1500.parquet")


@dataclass
class ExperimentConfig:
    seed: int = 42
    max_train_rows: Optional[int] = None  # Set e.g. 50000 for fast local tests; None for full Kaggle GPU
    dataset_dir: Optional[str] = None
    checkpoint_dir: Optional[str] = None
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


class PathResolver:
    """Dual-environment resolver: Explicit/ENV -> Kaggle GPU (+ slug scan) -> Local Full -> Repo Sample."""

    @staticmethod
    def _scan_kaggle_input(kaggle_root: Path = Path("/kaggle/input"), max_depth: int = 4) -> Optional[Path]:
        if not kaggle_root.is_dir():
            return None
        queue: List[Tuple[Path, int]] = [(kaggle_root, 0)]
        while queue:
            current, depth = queue.pop(0)
            if (current / "train" / "flows_tabular.parquet").is_file():
                return current
            if depth < max_depth:
                try:
                    children = sorted((p for p in current.iterdir() if p.is_dir()), key=lambda p: p.name)
                except OSError:
                    children = []
                for child in children:
                    queue.append((child, depth + 1))
        return None

    @classmethod
    def resolve(cls, config: ExperimentConfig) -> Dict[str, Any]:
        env_data = os.environ.get("TOPIC04_DATA_DIR", "").strip()
        env_ckpt = os.environ.get("TOPIC04_CHECKPOINT_DIR", "").strip()

        resolved_data: Optional[Path] = None
        mode = "unknown"
        is_sample = False

        # Resolve dataset path
        candidates: List[Tuple[str, Path]] = []
        if config.dataset_dir:
            candidates.append(("custom", Path(config.dataset_dir)))
        elif env_data:
            candidates.append(("custom", Path(env_data)))
        else:
            candidates.append(("kaggle", KAGGLE_DATASET_DIR))

        def _inspect(cand: Path) -> Optional[Tuple[Path, bool]]:
            if not cand.exists():
                return None
            if cand.is_file() and cand.suffix.lower() == ".parquet":
                return cand, True
            if cand.is_dir():
                if (cand / "train" / "flows_tabular.parquet").is_file():
                    return cand, False
                if (cand / "NF-UQ-NIDS-v2" / "train" / "flows_tabular.parquet").is_file():
                    return cand / "NF-UQ-NIDS-v2", False
                if (cand / "nfuq_sample_1500.parquet").is_file():
                    return cand / "nfuq_sample_1500.parquet", True
            return None

        for label, cand in candidates:
            hit = _inspect(cand)
            if hit is not None:
                resolved_data, is_sample = hit
                mode = label
                break

        if resolved_data is None and not config.dataset_dir and not env_data:
            kaggle_input_root = Path("/kaggle/input")
            if kaggle_input_root.exists():
                slug_dir = cls._scan_kaggle_input(kaggle_input_root, max_depth=4)
                if slug_dir is not None:
                    resolved_data, mode, is_sample = slug_dir, "kaggle_slug_scan", False

        if resolved_data is None and not config.dataset_dir and not env_data:
            for label, cand in (
                ("local_full", LOCAL_DATASET_DIR),
                ("local_sample", REPO_SAMPLE_PATH),
                ("local_sample_rel", Path("data/public_loaders/sample_data/nfuq_sample_1500.parquet")),
                ("local_sample_up", Path("../data/public_loaders/sample_data/nfuq_sample_1500.parquet")),
            ):
                hit = _inspect(cand)
                if hit is not None:
                    resolved_data, is_sample = hit
                    mode = label
                    break

        if resolved_data is None:
            raise FileNotFoundError("Could not locate NF-UQ-NIDS-v2 dataset (Kaggle, local, or sample).")

        # Resolve checkpoint directory
        if config.checkpoint_dir:
            ckpt_dir = Path(config.checkpoint_dir)
        elif env_ckpt:
            ckpt_dir = Path(env_ckpt)
        elif Path("/kaggle/working").exists():
            ckpt_dir = KAGGLE_CHECKPOINT_DIR
        elif LOCAL_CHECKPOINT_DIR.parent.exists():
            ckpt_dir = LOCAL_CHECKPOINT_DIR
        else:
            ckpt_dir = Path.cwd() / "Checkpoint"

        ckpt_dir.mkdir(parents=True, exist_ok=True)
        return {
            "mode": mode,
            "is_sample_mode": is_sample,
            "dataset_path": resolved_data,
            "checkpoint_dir": ckpt_dir,
        }


def set_global_seed(seed: int = 42) -> Dict[str, str]:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    return {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__,
        "torch": torch.__version__,
        "joblib": joblib.__version__,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }


CONFIG = ExperimentConfig(seed=42)
VERSIONS = set_global_seed(CONFIG.seed)
PATHS = PathResolver.resolve(CONFIG)
print("Resolved Paths:", {k: str(v) for k, v in PATHS.items()})
print("Reproducibility & Library Versions:", json.dumps({"seed": CONFIG.seed, **VERSIONS}, indent=2))

Resolved Paths: {'mode': 'kaggle_slug_scan', 'is_sample_mode': 'False', 'dataset_path': '/kaggle/input/datasets/vxdwkhoa/dataset-nf/NF-UQ-NIDS-v2', 'checkpoint_dir': '/kaggle/working/Checkpoint'}
Reproducibility & Library Versions: {
  "seed": 42,
  "python": "3.13.16",
  "numpy": "2.1.3",
  "pandas": "2.3.3",
  "scikit-learn": "1.6.1",
  "torch": "2.11.0+cu130",
  "joblib": "1.6.0",
  "device": "cuda"
}


## Cell 2 — Data Loader & No-L7 Privacy Validator

In [2]:
UNSEEN_ATTACK_FAMILIES: FrozenSet[str] = frozenset({"Worms", "Shellcode", "Ransomware"})


class PrivacyAndDataLoader:
    """
    Loads NF-UQ-NIDS-v2 Temporal + Host Holdout + Unseen Attack Holdout splits
    and enforces strict No-L7 Payload privacy validation (SR-01, CR-06, Section 13.4 Step 7).
    """

    FORBIDDEN_PATTERNS = [
        r"^http_", r"^ssh_command", r"^ssh_terminal", r"^ssh_keystroke", r"^ssh_auth_password",
        r"^decrypted_", r"^tls_session_key", r"^private_key", r"^raw_pcap", r"^payload",
        r"^l7_proto$", r"^dns_query_", r"^dns_ttl_", r"^ftp_command_",
        r"^ipv4_src_addr$", r"^ipv4_dst_addr$", r"^raw_s(ni|an)$",
    ]
    IPV4_RE = re.compile(r"^(?:[0-9]{1,3}\.){3}[0-9]{1,3}$")

    def __init__(self, paths: Dict[str, Any]) -> None:
        self.paths = paths

    def validate_dataframe_privacy(self, df: pd.DataFrame, split_name: str) -> Dict[str, Any]:
        violations = []
        for col in df.columns:
            col_l = col.lower()
            for pat in self.FORBIDDEN_PATTERNS:
                if re.search(pat, col_l):
                    violations.append(f"Forbidden L7/raw-IP column detected: {col}")
        for tok_col in ("src_host_token", "dst_host_token"):
            if tok_col in df.columns:
                unique_vals = df[tok_col].dropna().astype(str).unique()
                if any(self.IPV4_RE.match(v) for v in unique_vals):
                    violations.append(f"Raw IPv4 leaked inside {tok_col}")
        if violations:
            raise ValueError(f"Privacy Validator FAILED on {split_name}: {violations}")

        missing_features = [c for c in L3_L4_FEATURE_COLUMNS if c not in df.columns]
        if missing_features:
            raise ValueError(f"Dataset '{split_name}' missing required L3/L4 features: {missing_features}")

        return {"split": split_name, "passed_no_l7_audit": True, "rows": len(df), "violations": 0}

    @staticmethod
    def _sort_by_flow_order(df: pd.DataFrame) -> pd.DataFrame:
        if "flow_order_id" in df.columns and len(df) > 0:
            return df.sort_values("flow_order_id", kind="mergesort").reset_index(drop=True)
        return df.reset_index(drop=True)

    @classmethod
    def _split_sample_temporal(
        cls, df_raw: pd.DataFrame
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        if "attack_family" in df_raw.columns:
            unseen_mask = df_raw["attack_family"].astype(str).isin(UNSEEN_ATTACK_FAMILIES)
        else:
            unseen_mask = pd.Series(False, index=df_raw.index)

        df_unseen = cls._sort_by_flow_order(df_raw.loc[unseen_mask].copy())
        df_known = cls._sort_by_flow_order(df_raw.loc[~unseen_mask].copy())

        n = len(df_known)
        if n == 0:
            df_train = df_known.copy()
            df_val = df_known.iloc[0:0].copy()
            df_test_known = df_known.iloc[0:0].copy()
        elif n == 1:
            df_train = df_known.iloc[:1].reset_index(drop=True)
            df_val = df_known.iloc[0:0].copy()
            df_test_known = df_known.iloc[0:0].copy()
        elif n == 2:
            df_train = df_known.iloc[:1].reset_index(drop=True)
            df_val = df_known.iloc[1:2].reset_index(drop=True)
            df_test_known = df_known.iloc[0:0].copy()
        else:
            n_train = max(1, int(n * 0.70))
            n_val = max(1, int(n * 0.15))
            if n_train + n_val >= n:
                n_train = max(1, n - 2)
                n_val = 1
            df_train = df_known.iloc[:n_train].reset_index(drop=True)
            df_val = df_known.iloc[n_train:n_train + n_val].reset_index(drop=True)
            df_test_known = df_known.iloc[n_train + n_val:].reset_index(drop=True)

        df_test = cls._sort_by_flow_order(
            pd.concat([df_test_known, df_unseen], axis=0, ignore_index=True)
        )
        return df_train, df_val, df_test, df_unseen

    def load_splits(self, max_train_rows: Optional[int] = None) -> Dict[str, pd.DataFrame]:
        dataset_path: Path = self.paths["dataset_path"]
        if self.paths["is_sample_mode"]:
            df_raw = pd.read_parquet(dataset_path)
            self.validate_dataframe_privacy(df_raw, "sample_raw")
            df_train, df_val, df_test, df_unseen = self._split_sample_temporal(df_raw)
        else:
            df_train = pd.read_parquet(dataset_path / "train" / "flows_tabular.parquet")
            df_val = pd.read_parquet(dataset_path / "val" / "flows_tabular.parquet")
            df_test = pd.read_parquet(dataset_path / "test" / "flows_tabular.parquet")
            unseen_path = dataset_path / "test" / "unseen_attack_holdout.parquet"
            if unseen_path.is_file():
                df_unseen = pd.read_parquet(unseen_path)
            elif "attack_family" in df_test.columns:
                df_unseen = df_test[df_test["attack_family"].astype(str).isin(UNSEEN_ATTACK_FAMILIES)].copy()
            else:
                df_unseen = df_test.iloc[0:0].copy()

        if max_train_rows is not None and len(df_train) > max_train_rows:
            df_train = df_train.iloc[:max_train_rows].copy()

        splits = {"train": df_train, "val": df_val, "test": df_test, "unseen_holdout": df_unseen}
        for name, frame in splits.items():
            audit = self.validate_dataframe_privacy(frame, name)
            print(f"[PRIVACY PASS] {audit}")
        return splits


loader = PrivacyAndDataLoader(PATHS)
splits = loader.load_splits(max_train_rows=CONFIG.max_train_rows)

[PRIVACY PASS] {'split': 'train', 'passed_no_l7_audit': True, 'rows': 425771, 'violations': 0}
[PRIVACY PASS] {'split': 'val', 'passed_no_l7_audit': True, 'rows': 91230, 'violations': 0}
[PRIVACY PASS] {'split': 'test', 'passed_no_l7_audit': True, 'rows': 135108, 'violations': 0}
[PRIVACY PASS] {'split': 'unseen_holdout', 'passed_no_l7_audit': True, 'rows': 635, 'violations': 0}


## Cell 3 — Train-Only Feature Sanitization & Scaling

In [3]:
from sklearn.preprocessing import StandardScaler


class FlowFeatureTransformer:
    """
    Train-only feature sanitizer & scaler (SR-08 anti-leakage):
    1. Casts to float64 column-wise and replaces +inf/-inf (empirical overflow in *_second_bytes) with NaN.
    2. Imputes with Train-only finite medians and clips to Train-only quantile bounds.
    3. Fits StandardScaler strictly on Train (clamping scale_ < 1e-12 -> 1.0) and outputs finite float32 arrays.
    """

    def __init__(
        self,
        feature_cols: Sequence[str] = L3_L4_FEATURE_COLUMNS,
        clip_quantiles: Tuple[float, float] = (0.0, 0.9999),
        output_dtype: str = "float32",
    ) -> None:
        if len(feature_cols) == 0:
            raise ValueError("feature_cols must not be empty.")
        if len(clip_quantiles) != 2 or not (0.0 <= clip_quantiles[0] <= clip_quantiles[1] <= 1.0):
            raise ValueError(f"clip_quantiles must satisfy 0.0 <= low <= high <= 1.0, got {clip_quantiles}")
        if output_dtype not in ("float32", "float64"):
            raise ValueError(f"output_dtype must be 'float32' or 'float64', got '{output_dtype}'")

        self.feature_cols: Tuple[str, ...] = tuple(feature_cols)
        self.clip_quantiles: Tuple[float, float] = (float(clip_quantiles[0]), float(clip_quantiles[1]))
        self.output_dtype: str = output_dtype

        self.scaler_: StandardScaler = StandardScaler()
        self.medians_: Optional[np.ndarray] = None
        self.lower_bounds_: Optional[np.ndarray] = None
        self.upper_bounds_: Optional[np.ndarray] = None
        self.is_fitted_: bool = False

    def _validate_columns(self, df: pd.DataFrame) -> None:
        missing = [c for c in self.feature_cols if c not in df.columns]
        if missing:
            raise ValueError(f"DataFrame is missing required feature columns: {missing}")

    def _extract_float64(self, df: pd.DataFrame) -> np.ndarray:
        self._validate_columns(df)
        cols = list(self.feature_cols)
        arrays = []
        for col in cols:
            series = df[col]
            if pd.api.types.is_numeric_dtype(series):
                arr = series.to_numpy(dtype=np.float64, copy=True)
            else:
                arr = pd.to_numeric(series, errors="coerce").to_numpy(dtype=np.float64, copy=True)
            arrays.append(arr)
        X = np.column_stack(arrays).astype(np.float64, copy=False)
        return np.where(np.isfinite(X), X, np.nan)

    def fit(self, df_train: pd.DataFrame) -> "FlowFeatureTransformer":
        if len(df_train) == 0:
            raise ValueError("Cannot fit FlowFeatureTransformer on an empty DataFrame.")

        X = self._extract_float64(df_train)
        n_features = X.shape[1]
        medians = np.zeros(n_features, dtype=np.float64)
        for idx in range(n_features):
            col_vals = X[:, idx]
            finite_mask = np.isfinite(col_vals)
            medians[idx] = float(np.median(col_vals[finite_mask])) if np.any(finite_mask) else 0.0
        self.medians_ = medians

        nan_mask = np.isnan(X)
        if np.any(nan_mask):
            X = np.where(nan_mask, self.medians_, X)

        q_low, q_high = self.clip_quantiles
        self.lower_bounds_ = np.quantile(X, q_low, axis=0).astype(np.float64)
        self.upper_bounds_ = np.quantile(X, q_high, axis=0).astype(np.float64)
        X = np.clip(X, self.lower_bounds_, self.upper_bounds_)

        self.scaler_ = StandardScaler()
        self.scaler_.fit(X)
        self.scaler_.scale_ = np.where(self.scaler_.scale_ < 1e-12, 1.0, self.scaler_.scale_)
        self.is_fitted_ = True
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        if not self.is_fitted_:
            raise RuntimeError("FlowFeatureTransformer must be fitted on Train before transform().")
        self._validate_columns(df)

        out_dtype = np.dtype(self.output_dtype)
        if len(df) == 0:
            return np.empty((0, len(self.feature_cols)), dtype=out_dtype)

        X = self._extract_float64(df)
        nan_mask = np.isnan(X)
        if np.any(nan_mask):
            X = np.where(nan_mask, self.medians_, X)

        X = np.clip(X, self.lower_bounds_, self.upper_bounds_)
        X = (X - self.scaler_.mean_) / self.scaler_.scale_
        X_out = X.astype(out_dtype, copy=False)
        if not np.all(np.isfinite(X_out)):
            raise ValueError("Non-finite values detected after FlowFeatureTransformer.transform().")
        return X_out

    def fit_transform(self, df_train: pd.DataFrame) -> np.ndarray:
        return self.fit(df_train).transform(df_train)

    def get_state(self) -> Dict[str, Any]:
        if not self.is_fitted_:
            raise RuntimeError("Cannot serialize an unfitted FlowFeatureTransformer.")
        return {
            "feature_cols": list(self.feature_cols),
            "clip_quantiles": [float(self.clip_quantiles[0]), float(self.clip_quantiles[1])],
            "output_dtype": self.output_dtype,
            "medians": [float(x) for x in self.medians_],
            "lower_bounds": [float(x) for x in self.lower_bounds_],
            "upper_bounds": [float(x) for x in self.upper_bounds_],
            "mean": [float(x) for x in self.scaler_.mean_],
            "scale": [float(x) for x in self.scaler_.scale_],
            "var": [float(x) for x in self.scaler_.var_],
            "n_samples_seen": int(self.scaler_.n_samples_seen_),
        }

    @classmethod
    def from_state(cls, state: Dict[str, Any]) -> "FlowFeatureTransformer":
        obj = cls(
            feature_cols=state["feature_cols"],
            clip_quantiles=(float(state["clip_quantiles"][0]), float(state["clip_quantiles"][1])),
            output_dtype=str(state.get("output_dtype", "float32")),
        )
        obj.medians_ = np.asarray(state["medians"], dtype=np.float64)
        obj.lower_bounds_ = np.asarray(state["lower_bounds"], dtype=np.float64)
        obj.upper_bounds_ = np.asarray(state["upper_bounds"], dtype=np.float64)

        scaler = StandardScaler()
        scaler.mean_ = np.asarray(state["mean"], dtype=np.float64)
        scaler.scale_ = np.asarray(state["scale"], dtype=np.float64)
        scaler.scale_ = np.where(scaler.scale_ < 1e-12, 1.0, scaler.scale_)
        scaler.var_ = np.asarray(state["var"], dtype=np.float64)
        scaler.n_features_in_ = len(obj.feature_cols)
        scaler.n_samples_seen_ = int(state.get("n_samples_seen", 0))
        obj.scaler_ = scaler
        obj.is_fitted_ = True
        return obj


class TargetLabelEncoder:
    """Fits known `attack_family` classes on Train; maps unseen or missing labels safely to -1."""

    def __init__(self, unknown_index: int = -1, unknown_label: str = "UNKNOWN_UNSEEN") -> None:
        self.unknown_index: int = int(unknown_index)
        self.unknown_label: str = str(unknown_label)
        self.classes_: List[str] = []
        self.class_to_idx_: Dict[str, int] = {}
        self.idx_to_class_: Dict[int, str] = {}
        self.is_fitted_: bool = False

    def fit(self, y: Any) -> "TargetLabelEncoder":
        values = list(y)
        if len(values) == 0:
            raise ValueError("Cannot fit TargetLabelEncoder on empty labels.")
        unique_classes = sorted({str(v) for v in values if pd.notna(v)})
        if not unique_classes:
            raise ValueError("TargetLabelEncoder found no non-null label classes to fit.")
        self.classes_ = unique_classes
        self.class_to_idx_ = {c: i for i, c in enumerate(self.classes_)}
        self.idx_to_class_ = {i: c for i, c in enumerate(self.classes_)}
        self.is_fitted_ = True
        return self

    def transform(self, y: Any) -> np.ndarray:
        if not self.is_fitted_:
            raise RuntimeError("TargetLabelEncoder must be fitted before calling transform().")
        encoded = [
            self.class_to_idx_.get(str(v), self.unknown_index) if pd.notna(v) else self.unknown_index
            for v in y
        ]
        return np.asarray(encoded, dtype=np.int64)

    def fit_transform(self, y: Any) -> np.ndarray:
        return self.fit(y).transform(y)

    def inverse_transform(self, indices: Sequence[int]) -> List[str]:
        if not self.is_fitted_:
            raise RuntimeError("TargetLabelEncoder must be fitted before calling inverse_transform().")
        return [self.idx_to_class_.get(int(idx), self.unknown_label) for idx in indices]

    def known_mask(self, y_encoded: Sequence[int]) -> np.ndarray:
        return np.asarray(y_encoded, dtype=np.int64) != self.unknown_index

    def get_state(self) -> Dict[str, Any]:
        if not self.is_fitted_:
            raise RuntimeError("Cannot serialize an unfitted TargetLabelEncoder.")
        return {
            "classes": list(self.classes_),
            "unknown_index": self.unknown_index,
            "unknown_label": self.unknown_label,
        }

    @classmethod
    def from_state(cls, state: Dict[str, Any]) -> "TargetLabelEncoder":
        obj = cls(
            unknown_index=int(state.get("unknown_index", -1)),
            unknown_label=str(state.get("unknown_label", "UNKNOWN_UNSEEN")),
        )
        obj.classes_ = [str(c) for c in state["classes"]]
        obj.class_to_idx_ = {c: i for i, c in enumerate(obj.classes_)}
        obj.idx_to_class_ = {i: c for i, c in enumerate(obj.classes_)}
        obj.is_fitted_ = True
        return obj


transformer = FlowFeatureTransformer().fit(splits["train"])
label_encoder = TargetLabelEncoder().fit(splits["train"]["attack_family"])
print(f"Transformer fitted on Train ({len(splits['train']):,} rows, {len(L3_L4_FEATURE_COLUMNS)} features).")
print(f"Known Train attack_family classes ({len(label_encoder.classes_)}): {label_encoder.classes_[:10]}...")

Transformer fitted on Train (425,771 rows, 40 features).
Known Train attack_family classes (19): ['Analysis', 'Backdoor', 'Benign', 'Bot', 'Brute Force', 'DDoS', 'DoS', 'Exploits', 'Fuzzers', 'Generic']...


## Cell 4 — TemporalSequenceDataset (Pure L3/L4 Host-Ordered Windows)

In [4]:
from torch.utils.data import Dataset, DataLoader


class TemporalSequenceDataset(Dataset):
    """
    Constructs 3D temporal flow sequences `(N, seq_len, 40)` grouped strictly by
    `src_host_token` and sorted chronologically by `flow_order_id` (no graph features, no padding).
    """

    def __init__(
        self,
        df: pd.DataFrame,
        transformer: FlowFeatureTransformer,
        seq_len: int = 8,
        stride: int = 4,
    ) -> None:
        if seq_len < 1:
            raise ValueError(f"seq_len must be >= 1, got {seq_len}")
        if stride < 1:
            raise ValueError(f"stride must be >= 1, got {stride}")

        self.seq_len = int(seq_len)
        self.stride = int(stride)
        self.num_hosts = 0
        self.num_active_hosts = 0
        self.num_skipped_hosts = 0

        if len(df) == 0:
            self.X = torch.empty((0, self.seq_len, len(L3_L4_FEATURE_COLUMNS)), dtype=torch.float32)
            self.y = torch.empty((0,), dtype=torch.long)
            self.num_sequences = 0
            warnings.warn("TemporalSequenceDataset received an empty DataFrame; returning 0 sequences.", UserWarning)
            return

        X_all = transformer.transform(df)
        y_all = pd.to_numeric(df["label"], errors="raise").to_numpy(dtype=np.int64)

        meta_df = df[["src_host_token", "flow_order_id"]].copy()
        meta_df["_pos"] = np.arange(len(df), dtype=np.int64)
        meta_sorted = meta_df.sort_values(["src_host_token", "flow_order_id"], kind="mergesort").reset_index(drop=True)

        sort_pos = meta_sorted["_pos"].to_numpy(dtype=np.int64)
        X_sorted = X_all[sort_pos]
        y_sorted = y_all[sort_pos]

        grouped = meta_sorted.groupby("src_host_token", sort=False, observed=True)
        X_seqs: List[np.ndarray] = []
        y_seqs: List[int] = []

        for _, idx_arr in grouped.indices.items():
            n = len(idx_arr)
            self.num_hosts += 1
            if n < self.seq_len:
                self.num_skipped_hosts += 1
                continue
            self.num_active_hosts += 1
            for start in range(0, n - self.seq_len + 1, self.stride):
                win_idx = idx_arr[start:start + self.seq_len]
                X_seqs.append(X_sorted[win_idx])
                y_seqs.append(int(np.max(y_sorted[win_idx])))

        if X_seqs:
            stacked = np.stack(X_seqs, axis=0).astype(np.float32, copy=False)
            self.X = torch.from_numpy(stacked).to(dtype=torch.float32)
            self.y = torch.tensor(y_seqs, dtype=torch.long)
        else:
            self.X = torch.empty((0, self.seq_len, len(L3_L4_FEATURE_COLUMNS)), dtype=torch.float32)
            self.y = torch.empty((0,), dtype=torch.long)
            warnings.warn(
                f"TemporalSequenceDataset produced 0 sequences for seq_len={self.seq_len}, stride={self.stride}.",
                UserWarning,
            )
        self.num_sequences = int(self.X.shape[0])

    def __len__(self) -> int:
        return self.num_sequences

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.X[index], self.y[index]


SEQ_LEN = 4 if PATHS["is_sample_mode"] else 8
STRIDE = 2 if PATHS["is_sample_mode"] else 4

train_seq_ds = TemporalSequenceDataset(splits["train"], transformer, seq_len=SEQ_LEN, stride=STRIDE)
val_seq_ds = TemporalSequenceDataset(splits["val"], transformer, seq_len=SEQ_LEN, stride=STRIDE)
test_seq_ds = TemporalSequenceDataset(splits["test"], transformer, seq_len=SEQ_LEN, stride=STRIDE)
unseen_seq_ds = TemporalSequenceDataset(splits["unseen_holdout"], transformer, seq_len=SEQ_LEN, stride=STRIDE)
print(
    f"Temporal Sequences (seq_len={SEQ_LEN}, stride={STRIDE}) -> "
    f"Train: {len(train_seq_ds):,}, Val: {len(val_seq_ds):,}, Test: {len(test_seq_ds):,}, Unseen: {len(unseen_seq_ds):,}"
)

Temporal Sequences (seq_len=8, stride=4) -> Train: 105,185, Val: 22,023, Test: 32,902, Unseen: 153


## Cell 5 — Pure PyTorch LSTM Model Definition

In [5]:
import torch.nn as nn
from torch.utils.data import TensorDataset


class PureLSTMNetwork(nn.Module):
    """Pure PyTorch nn.LSTM temporal sequence classifier (40 L3/L4 features, no graph layers)."""

    def __init__(self, input_dim: int = 40, hidden_dim: int = 64, num_layers: int = 2, dropout: float = 0.2) -> None:
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_dim = int(hidden_dim)
        self.num_layers = int(num_layers)
        self.dropout = float(dropout) if self.num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=self.input_dim,
            hidden_size=self.hidden_dim,
            num_layers=self.num_layers,
            batch_first=True,
            dropout=self.dropout,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(self.hidden_dim),
            nn.Dropout(float(dropout)),
            nn.Linear(self.hidden_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


class LSTMSequenceModel:
    """OOP wrapper encapsulating PureLSTMNetwork, device management, and sequence metadata."""

    def __init__(
        self,
        input_dim: int = 40,
        hidden_dim: int = 64,
        num_layers: int = 2,
        dropout: float = 0.2,
        seq_len: int = 8,
        stride: int = 4,
        threshold: float = 0.5,
        device: str = "cpu",
    ) -> None:
        self.model_name = "week7_lstm_sequence"
        self.task_type = "sequence_binary"
        self.input_dim = int(input_dim)
        self.hidden_dim = int(hidden_dim)
        self.num_layers = int(num_layers)
        self.dropout = float(dropout)
        self.seq_len = int(seq_len)
        self.stride = int(stride)
        self.threshold = float(threshold)
        self.device = torch.device(device)
        self.is_fitted = False
        self.network = PureLSTMNetwork(self.input_dim, self.hidden_dim, self.num_layers, self.dropout).to(self.device)

    def state_dict(self) -> Dict[str, Any]:
        return self.network.state_dict()

    def load_state_dict(self, state_dict: Dict[str, Any]) -> None:
        self.network.load_state_dict(state_dict)
        self.network.to(self.device)
        self.network.eval()
        self.is_fitted = True

    def fit(
        self,
        X: Any,
        y: Optional[Any] = None,
        epochs: int = 5,
        batch_size: int = 256,
        lr: float = 1e-3,
        seed: int = 42,
    ) -> "LSTMSequenceModel":
        trainer = LSTMTrainer(model=self, epochs=epochs, batch_size=batch_size, lr=lr, seed=seed)
        if y is not None and (isinstance(X, (np.ndarray, torch.Tensor, list, tuple)) or not hasattr(X, "__getitem__")):
            ds = TensorDataset(
                torch.as_tensor(X, dtype=torch.float32).detach().cpu(),
                torch.as_tensor(y, dtype=torch.float32).detach().cpu(),
            )
            trainer.fit(ds)
        else:
            trainer.fit(X)
        return self

    @torch.no_grad()
    def predict_proba(self, dataset: Any, batch_size: int = 256) -> np.ndarray:
        self.network.eval()
        if isinstance(dataset, DataLoader):
            loader = dataset
        else:
            use_cuda = self.device.type == "cuda"
            is_cpu_ds = True
            if len(dataset) > 0:
                s0 = dataset[0]
                x0 = s0[0] if isinstance(s0, (list, tuple)) else s0
                if isinstance(x0, torch.Tensor) and x0.is_cuda:
                    is_cpu_ds = False
            loader = DataLoader(
                dataset,
                batch_size=batch_size,
                shuffle=False,
                num_workers=0,
                pin_memory=(use_cuda and is_cpu_ds),
            )
        non_blocking = self.device.type == "cuda"
        probs_list = []
        for batch in loader:
            xb = batch[0] if isinstance(batch, (list, tuple)) else batch
            logits = self.network(xb.to(self.device, dtype=torch.float32, non_blocking=non_blocking))
            p = torch.sigmoid(logits).detach().cpu().numpy()
            probs_list.append(p.reshape(-1).astype(np.float64))
        return np.concatenate(probs_list, axis=0) if probs_list else np.zeros((0,), dtype=np.float64)

    def predict(self, dataset: Any, threshold: Optional[float] = None, batch_size: int = 256) -> np.ndarray:
        thr = float(threshold) if threshold is not None else self.threshold
        return (self.predict_proba(dataset, batch_size=batch_size) >= thr).astype(np.int64)

    def get_metadata(self) -> Dict[str, Any]:
        return {
            "model_name": self.model_name,
            "task_type": self.task_type,
            "input_dim": self.input_dim,
            "hidden_dim": self.hidden_dim,
            "num_layers": self.num_layers,
            "dropout": self.dropout,
            "seq_len": self.seq_len,
            "stride": self.stride,
            "threshold": self.threshold,
            "device": str(self.device),
            "graph_features_used": False,
            "is_fitted": self.is_fitted,
        }

## Cell 6 — LSTMTrainer, LSTMEvaluator & CheckpointManager

In [6]:
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score


class LSTMTrainer:
    """Deterministic PyTorch trainer supporting Kaggle CUDA GPU and local CPU."""

    def __init__(
        self,
        model: LSTMSequenceModel,
        epochs: int = 5,
        batch_size: int = 256,
        lr: float = 1e-3,
        seed: int = 42,
    ) -> None:
        self.model = model
        self.epochs = int(epochs)
        self.batch_size = int(batch_size)
        self.lr = float(lr)
        self.seed = int(seed)

    @staticmethod
    def _extract_labels(dataset: Any) -> np.ndarray:
        if hasattr(dataset, "y"):
            if dataset.y is None:
                raise ValueError("Dataset labels (`dataset.y`) cannot be None for LSTM training.")
            yv = dataset.y
            return yv.detach().cpu().numpy().astype(np.float32, copy=False) if isinstance(yv, torch.Tensor) else np.asarray(yv, dtype=np.float32)
        if hasattr(dataset, "tensors") and len(getattr(dataset, "tensors", ())) >= 2:
            yv = dataset.tensors[1]
            return yv.detach().cpu().numpy().astype(np.float32, copy=False) if isinstance(yv, torch.Tensor) else np.asarray(yv, dtype=np.float32)
        return np.asarray([float(dataset[i][1]) for i in range(len(dataset))], dtype=np.float32)

    def fit(self, train_ds: Any, val_ds: Optional[Any] = None) -> Dict[str, Any]:
        set_global_seed(self.seed)
        if len(train_ds) == 0:
            raise ValueError("Cannot train LSTM on an empty TemporalSequenceDataset.")

        use_cuda = self.model.device.type == "cuda"
        s0 = train_ds[0]
        x0 = s0[0] if isinstance(s0, (list, tuple)) else s0
        is_cpu_train = not (isinstance(x0, torch.Tensor) and x0.is_cuda)

        gen = torch.Generator().manual_seed(self.seed)
        train_loader = DataLoader(
            train_ds,
            batch_size=self.batch_size,
            shuffle=True,
            generator=gen,
            num_workers=0,
            pin_memory=(use_cuda and is_cpu_train),
        )

        y_train = self._extract_labels(train_ds)
        n_pos = float(np.sum(y_train == 1.0))
        n_neg = float(np.sum(y_train == 0.0))
        if n_pos > 0 and n_neg > 0:
            pw = torch.tensor([min(n_neg / n_pos, 20.0)], dtype=torch.float32, device=self.model.device)
            criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
        else:
            criterion = nn.BCEWithLogitsLoss()

        optimizer = torch.optim.AdamW(self.model.network.parameters(), lr=self.lr, weight_decay=1e-4)
        history: Dict[str, List[float]] = {"train_loss": [], "val_loss": []}

        t0 = time.perf_counter()
        for _ in range(self.epochs):
            self.model.network.train()
            tr_loss = 0.0
            for xb, yb in train_loader:
                xb = xb.to(self.model.device, dtype=torch.float32, non_blocking=use_cuda)
                yb = yb.to(self.model.device, dtype=torch.float32, non_blocking=use_cuda)
                optimizer.zero_grad()
                loss = criterion(self.model.network(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.network.parameters(), 1.0)
                optimizer.step()
                tr_loss += float(loss.item())
            history["train_loss"].append(tr_loss / max(1, len(train_loader)))

            if val_ds is not None and len(val_ds) > 0:
                vs0 = val_ds[0]
                vx0 = vs0[0] if isinstance(vs0, (list, tuple)) else vs0
                is_cpu_val = not (isinstance(vx0, torch.Tensor) and vx0.is_cuda)
                val_loader = DataLoader(
                    val_ds,
                    batch_size=self.batch_size * 2,
                    shuffle=False,
                    num_workers=0,
                    pin_memory=(use_cuda and is_cpu_val),
                )
                self.model.network.eval()
                v_loss = 0.0
                with torch.no_grad():
                    for xb, yb in val_loader:
                        xb = xb.to(self.model.device, dtype=torch.float32, non_blocking=use_cuda)
                        yb = yb.to(self.model.device, dtype=torch.float32, non_blocking=use_cuda)
                        v_loss += float(criterion(self.model.network(xb), yb).item())
                history["val_loss"].append(v_loss / max(1, len(val_loader)))

        self.model.is_fitted = True
        self.model.network.eval()
        return {**history, "train_seconds": float(time.perf_counter() - t0)}


class LSTMEvaluator:
    """Computes sequence classification metrics, seq_len/stride metadata, and CUDA-synchronized latency."""

    @staticmethod
    def evaluate(model: LSTMSequenceModel, ds: TemporalSequenceDataset, batch_size: int = 256) -> Dict[str, Any]:
        if getattr(ds, "y", None) is None:
            raise ValueError("Dataset labels (`ds.y`) cannot be None for LSTM evaluation.")
        if len(ds) == 0:
            return {
                "roc_auc": 0.5,
                "pr_auc": 0.0,
                "f1": 0.0,
                "precision": 0.0,
                "recall": 0.0,
                "fpr": 0.0,
                "confusion_matrix": [[0, 0], [0, 0]],
                "seq_len": ds.seq_len,
                "stride": ds.stride,
                "n_sequences": 0,
                "inference_latency_ms_per_seq": 0.0,
                "events_per_sec": 0.0,
            }

        is_cuda = model.device.type == "cuda"
        if is_cuda:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        probs = model.predict_proba(ds, batch_size=batch_size)
        if is_cuda:
            torch.cuda.synchronize()
        elapsed = max(time.perf_counter() - t0, 1e-9)

        preds = (probs >= getattr(model, "threshold", 0.5)).astype(np.int64)
        yt = ds.y.detach().cpu().numpy().astype(np.int64, copy=False) if isinstance(ds.y, torch.Tensor) else np.asarray(ds.y, dtype=np.int64)

        roc = float(roc_auc_score(yt, probs)) if len(np.unique(yt)) > 1 else 0.5
        pr = float(average_precision_score(yt, probs)) if len(np.unique(yt)) > 1 else float(np.mean(yt == 1))
        cm = confusion_matrix(yt, preds, labels=[0, 1])
        tn, fp, fn, tp = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])
        fpr = float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0

        return {
            "roc_auc": roc,
            "pr_auc": pr,
            "f1": float(f1_score(yt, preds, zero_division=0)),
            "precision": float(precision_score(yt, preds, zero_division=0)),
            "recall": float(recall_score(yt, preds, zero_division=0)),
            "fpr": fpr,
            "confusion_matrix": cm.tolist(),
            "seq_len": ds.seq_len,
            "stride": ds.stride,
            "n_sequences": int(len(yt)),
            "inference_latency_ms_per_seq": float((elapsed / max(1, len(yt))) * 1000.0),
            "events_per_sec": float(len(yt) / elapsed),
        }


class CheckpointManager:
    """Saves CPU-detached `.pt` state_dict + sanitized `_metadata.json` (`allow_nan=False`)."""

    def __init__(self, ckpt_dir: Path) -> None:
        self.ckpt_dir = Path(ckpt_dir)
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)

    @classmethod
    def _sanitize_json(cls, obj: Any) -> Any:
        if obj is None or isinstance(obj, (bool, str, int)):
            return obj
        if isinstance(obj, float):
            return obj if math.isfinite(obj) else None
        if isinstance(obj, Path):
            return str(obj)
        if isinstance(obj, np.bool_):
            return bool(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            val = float(obj)
            return val if math.isfinite(val) else None
        if isinstance(obj, np.ndarray):
            return cls._sanitize_json(obj.tolist())
        if isinstance(obj, Mapping):
            return {str(k): cls._sanitize_json(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple, set, frozenset)):
            return [cls._sanitize_json(v) for v in obj]
        return str(obj)

    def save_pt(self, name: str, state_dict: Dict[str, Any], metadata: Dict[str, Any]) -> Tuple[Path, Path]:
        pt_path = self.ckpt_dir / f"{name}.pt"
        meta_path = self.ckpt_dir / f"{name}_metadata.json"
        cpu_state = {
            k: (v.detach().cpu() if isinstance(v, torch.Tensor) else v)
            for k, v in state_dict.items()
        }
        torch.save(cpu_state, pt_path)
        clean_meta = self._sanitize_json(metadata)
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(clean_meta, f, indent=2, ensure_ascii=False, allow_nan=False)
        return pt_path, meta_path

## Cell 7 — Training, Sequence Evaluation, Latency & PyTorch Checkpointing

In [7]:
epochs = 2 if PATHS["is_sample_mode"] else 15
hidden_dim = 32 if PATHS["is_sample_mode"] else 64
num_layers = 1 if PATHS["is_sample_mode"] else 2
batch_size = 64 if PATHS["is_sample_mode"] else 256

lstm_model = LSTMSequenceModel(
    input_dim=len(L3_L4_FEATURE_COLUMNS),
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    dropout=0.2,
    seq_len=SEQ_LEN,
    stride=STRIDE,
    device=CONFIG.device,
)
history = LSTMTrainer(lstm_model, epochs=epochs, batch_size=batch_size, lr=1e-3, seed=CONFIG.seed).fit(train_seq_ds, val_seq_ds)
print("=== TRAINING HISTORY ===")
print(json.dumps(history, indent=2))

val_metrics = LSTMEvaluator.evaluate(lstm_model, val_seq_ds, batch_size=batch_size) if len(val_seq_ds) > 0 else None
lstm_metrics = LSTMEvaluator.evaluate(lstm_model, test_seq_ds, batch_size=batch_size)
unseen_metrics = LSTMEvaluator.evaluate(lstm_model, unseen_seq_ds, batch_size=batch_size) if len(unseen_seq_ds) > 0 else None

print("\n=== WEEK 7 PURE PYTORCH LSTM SEQUENCE METRICS (TEST) ===")
print(json.dumps(lstm_metrics, indent=2))
if val_metrics is not None:
    print("\n=== VALIDATION METRICS ===")
    print(json.dumps(val_metrics, indent=2))

ckpt_mgr = CheckpointManager(PATHS["checkpoint_dir"])
pt_path, meta_path = ckpt_mgr.save_pt(
    "week7_lstm_temporal",
    lstm_model.state_dict(),
    {
        "seed": CONFIG.seed,
        "device": CONFIG.device,
        "versions": VERSIONS,
        "environment": VERSIONS,
        "feature_columns": L3_L4_FEATURE_COLUMNS,
        "seq_len": SEQ_LEN,
        "stride": STRIDE,
        "batch_size": batch_size,
        "hidden_dim": hidden_dim,
        "num_layers": num_layers,
        "graph_features_used": False,
        "metrics": lstm_metrics,
        "val_metrics": val_metrics,
        "unseen_metrics": unseen_metrics,
        "transformer_state": transformer.get_state(),
        "saved_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    },
)

# Verify exact reload parity + FlowFeatureTransformer state round-trip
orig_probs = lstm_model.predict_proba(test_seq_ds, batch_size=batch_size)
reloaded_model = LSTMSequenceModel(
    input_dim=len(L3_L4_FEATURE_COLUMNS),
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    dropout=0.2,
    seq_len=SEQ_LEN,
    stride=STRIDE,
    device=CONFIG.device,
)
try:
    loaded_pt = torch.load(pt_path, map_location="cpu", weights_only=True)
except TypeError:
    loaded_pt = torch.load(pt_path, map_location="cpu")
state_dict = loaded_pt["state_dict"] if isinstance(loaded_pt, dict) and "state_dict" in loaded_pt else loaded_pt
reloaded_model.load_state_dict(state_dict)

with open(meta_path, "r", encoding="utf-8") as f:
    loaded_meta = json.load(f)
assert "transformer_state" in loaded_meta
reloaded_tf = FlowFeatureTransformer.from_state(loaded_meta["transformer_state"])
assert np.allclose(transformer.transform(splits["test"]), reloaded_tf.transform(splits["test"]), atol=1e-6)
assert np.allclose(orig_probs, reloaded_model.predict_proba(test_seq_ds, batch_size=batch_size), atol=1e-5)
print(f"\n[VERIFIED] Saved & reloaded PyTorch checkpoint + transformer_state: {pt_path.name}")

=== TRAINING HISTORY ===
{
  "train_loss": [
    0.007859757560398204,
    0.0006970476099495876,
    0.0005067746972084106,
    0.0005789319077155999,
    0.0002602934320544733,
    0.0002493050676211127,
    0.00019559928544139794,
    0.00025059883960781105,
    0.0001487616606954784,
    0.00010463252475411136,
    0.00010868449949223752,
    0.00018290988180927182,
    0.00015504532974875402,
    6.740295659262574e-05,
    9.688556899828785e-05
  ],
  "val_loss": [
    0.0041537877168593695,
    0.0021674493762640686,
    0.003946765701791513,
    0.001917936115040158,
    0.0010442189708018386,
    0.001589133269176686,
    0.0035832317616433847,
    0.00168826675735545,
    0.0014851926254870953,
    0.0012876639974618602,
    0.0018682397762652479,
    0.0016678555485379382,
    0.0010648959138925627,
    0.0015265326253199873,
    0.0010144764464609
  ],
  "train_seconds": 36.715668490999974
}

=== WEEK 7 PURE PYTORCH LSTM SEQUENCE METRICS (TEST) ===
{
  "roc_auc": 0.999998572